# Silver-to-Gold: Carregando o Data Warehouse

## Objetivos

- Ler os dados da camada Silver (tabela `escolas`).
- Popular as tabelas de dimensão.
- Popular a tabela fato (`fato_escola`) e a tabela bridge (`bridge_escola_etapa`).
- Garantir a idempotência do processo de carga.

#### Configuração do Spark

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("Silver to Gold ETL")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
    )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/24 20:46:18 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.1.41 instead (on interface wlp63s0)
25/11/24 20:46:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/lucas/fga/bancos2/grupo-18-bancos-2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lucas/.ivy2.5.2/cache
The jars for the packages stored in: /home/lucas/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-98326e0d-0b7f-4a4d-8ff7-1e52b72e9380;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 69ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframewo

#### Configuração do Banco de Dados

In [2]:
postgres_host = os.getenv("POSTGRES_HOST", "localhost")
postgres_port = os.getenv("POSTGRES_PORT", "5432")
postgres_db = os.getenv("POSTGRES_DB", "inep_db")
postgres_user = os.getenv("POSTGRES_USER", "inep")
postgres_password = os.getenv("POSTGRES_PASSWORD", "inep")
silver_table = os.getenv("POSTGRES_TABLE", "esc")
silver_schema = os.getenv("POSTGRES_SILVER_SCHEMA", "silver")
postgres_schema = os.getenv("POSTGRES_DW_SCHEMA", "dw")

dw_jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}?currentSchema={postgres_schema}"
silver_jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}?currentSchema={silver_schema}"

jdbc_properties = {
    "user": postgres_user,
    "password": postgres_password,
    "driver": "org.postgresql.Driver"
}

print(f"Conectando ao banco de dados: {silver_jdbc_url}")

Conectando ao banco de dados: jdbc:postgresql://localhost:5432/inep_db?currentSchema=silver


#### Carregar Dados da Camada Silver

In [3]:
print(f"Lendo dados da tabela silver: '{silver_table}'")

df_silver = (
    spark.read.jdbc(url=silver_jdbc_url, table=silver_table, properties=jdbc_properties)
    .withColumn("is_rur", F.col("is_rur").cast("boolean"))
    .withColumn("is_pub", F.col("is_pub").cast("boolean"))
)

df_silver.cache()

print(f"Total de registros lidos: {df_silver.count():,}")
df_silver.printSchema()

Lendo dados da tabela silver: 'esc'


Total de registros lidos: 156,423
root
 |-- cod_inep: long (nullable = true)
 |-- nom_esc: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- mun: string (nullable = true)
 |-- reg: string (nullable = true)
 |-- lca: string (nullable = true)
 |-- is_rur: boolean (nullable = true)
 |-- dpd_adm: string (nullable = true)
 |-- is_pub: boolean (nullable = true)
 |-- prt_esc: string (nullable = true)
 |-- prt_num: integer (nullable = true)
 |-- etp_mod: string (nullable = true)
 |-- qtd_etp: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- rst_atn: string (nullable = true)



## 1. Popular Tabelas de Dimensão

In [5]:
from psycopg2 import connect, sql

def _qualified_table(table_name: str):
    """Retorna o identificador qualificado para schema.tabela."""
    if "." in table_name:
        schema, table = table_name.split(".", 1)
        return sql.SQL("{}.{}").format(sql.Identifier(schema), sql.Identifier(table))
    return sql.Identifier(table_name)


def write_rows_individually(df, table_name, truncate=True, cascade=False):
    """
    Insere os registros de um DataFrame linha a linha na tabela alvo.
    Em caso de falha, exibe o índice e o conteúdo da linha problemática.
    """
    columns = df.columns
    if not columns:
        print(f"Nenhuma coluna disponível para inserir em '{table_name}'.")
        return

    conn = connect(
        host=postgres_host,
        port=postgres_port,
        dbname=postgres_db,
        user=postgres_user,
        password=postgres_password
    )
    conn.autocommit = False
    cursor = conn.cursor()

    qualified_table = _qualified_table(table_name)
    column_identifiers = sql.SQL(", ").join(sql.Identifier(col) for col in columns)
    placeholders = sql.SQL(", ").join(sql.Placeholder() for _ in columns)
    insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES ({})").format(
        qualified_table, column_identifiers, placeholders
    )

    try:
        if truncate:
            truncate_stmt = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY").format(
                qualified_table
            )
            if cascade:
                truncate_stmt += sql.SQL(" CASCADE")
            cursor.execute(truncate_stmt)
            conn.commit()
            print(f"Tabela '{table_name}' truncada antes da nova carga.")

        row_count = 0
        for row_count, row in enumerate(df.toLocalIterator(), start=1):
            values = [row[col] for col in columns]
            try:
                cursor.execute(insert_stmt, values)
                conn.commit()
            except Exception as err:
                conn.rollback()
                print(
                    f"Falha ao inserir linha {row_count} em '{table_name}': {row.asDict()}"
                )
                raise err

        print(f"Tabela '{table_name}' carregada com {row_count:,} registros.")
    finally:
        cursor.close()
        conn.close()

def load_dimension(
    df, table_name, select_cols, distinct=True, rename_map=None, drop_nulls=False
):
    """Função para extrair, transformar e carregar uma tabela de dimensão."""
    print(f"Processando dimensão: {table_name}")
    dim_df = df.select(*select_cols)

    if distinct:
        dim_df = dim_df.distinct()

    if rename_map:
        for old_name, new_name in rename_map.items():
            dim_df = dim_df.withColumnRenamed(old_name, new_name)

    if drop_nulls:
        dim_df = dim_df.dropna(subset=dim_df.columns)

    write_rows_individually(dim_df, table_name, cascade=True)

    # Retorna o DF lido do DW para obter os SKs gerados
    return spark.read.jdbc(url=dw_jdbc_url, table=table_name, properties=jdbc_properties)

#### 1.1. dim_localidade

In [6]:
df_loc = df_silver.withColumn("is_rur", F.col("is_rur").cast("boolean"))

dim_loc_srk = load_dimension(
    df=df_loc,
    table_name="dw.dim_loc",
    select_cols=["uf", "mun", "reg", "lca", "is_rur", "lat", "lon"]
)

Processando dimensão: dw.dim_loc
Tabela 'dw.dim_loc' truncada antes da nova carga.
Tabela 'dw.dim_loc' carregada com 156,184 registros.


#### 1.2. dim_dependencia

In [7]:
df_dpd = df_silver.withColumn("is_pub", F.col("is_pub").cast("boolean"))

dim_dpd_srk = load_dimension(
    df=df_dpd,
    table_name="dw.dim_dpd",
    select_cols=["dpd_adm", "is_pub"]
)

Processando dimensão: dw.dim_dpd
Tabela 'dw.dim_dpd' truncada antes da nova carga.
Tabela 'dw.dim_dpd' carregada com 4 registros.


#### 1.3. dim_porte

In [8]:
dim_prt_srk = load_dimension(
    df=df_silver,
    table_name="dw.dim_prt",
    select_cols=["prt_esc", "prt_num"],
    drop_nulls=True
)

Processando dimensão: dw.dim_prt
Tabela 'dw.dim_prt' truncada antes da nova carga.
Tabela 'dw.dim_prt' carregada com 5 registros.


#### 1.4. dim_restricao_atendimento

In [9]:
dim_rst_srk = load_dimension(
    df=df_silver,
    table_name="dw.dim_rst_atn",
    select_cols=["rst_atn"],
    rename_map={"rst_atn": "rst_desc"}
)

Processando dimensão: dw.dim_rst_atn
Tabela 'dw.dim_rst_atn' truncada antes da nova carga.
Tabela 'dw.dim_rst_atn' carregada com 5 registros.


#### 1.5. dim_etapa

In [10]:
df_etp = df_silver.select(
    F.explode(F.split(F.col("etp_mod"), ",")).alias("etp")
).withColumn("etp", F.trim(F.col("etp"))).distinct()

dim_etp_srk = load_dimension(
    df=df_etp,
    table_name="dw.dim_etp",
    select_cols=["etp"],
    distinct=False
)

Processando dimensão: dw.dim_etp
Tabela 'dw.dim_etp' truncada antes da nova carga.
Tabela 'dw.dim_etp' carregada com 5 registros.


## 2. Popular Tabela Fato e Bridge

In [16]:
df_fato_base = (
    df_silver.join(dim_loc_srk, ["uf", "mun", "reg", "lca", "is_rur", "lat", "lon"], "inner")
    .join(dim_dpd_srk, ["dpd_adm", "is_pub"], "inner")
    .join(dim_prt_srk, ["prt_esc", "prt_num"], "left")
    .join(dim_rst_srk, F.col("rst_atn") == F.col("rst_desc"), "left")
)

print("Joins com dimensões concluídos.")

Joins com dimensões concluídos.


In [19]:
df_fato = df_fato_base.select(
    "srk_loc",
    "srk_dpd",
    "srk_prt",
    "srk_rst",
    "cod_inep",
    "nom_esc"
).distinct()

write_rows_individually(df_fato, "dw.fat_esc", cascade=True)

print(f"Tabela 'dw.fat_esc' carregada com {df_fato.count():,} registros.")

Tabela 'dw.fat_esc' truncada antes da nova carga.


Tabela 'dw.fat_esc' carregada com 156,423 registros.


Tabela 'dw.fat_esc' carregada com 156,423 registros.


In [27]:
df_bridge_base = df_silver.select(
    "cod_inep",
    F.explode(F.split(F.col("etp_mod"), ",")).alias("etp")
).withColumn("etp", F.trim(F.col("etp")))

df_bridge = (
    df_bridge_base.join(dim_etp_srk, "etp", "inner")
    .select("cod_inep", "srk_etp")
    .distinct()
).withColumnRenamed("cod_inep", "srk_esc")

write_rows_individually(df_bridge, "dw.bridge_esc_etp", cascade=True)

print(f"Tabela 'dw.bridge_esc_etp' carregada com {df_bridge.count():,} registros.")

+--------+-------+
|srk_esc |srk_etp|
+--------+-------+
|53009649|1      |
|53008057|1      |
|53002016|1      |
|52283364|1      |
|52277356|1      |
|52132200|1      |
|52097765|1      |
|52083837|1      |
|52072142|1      |
|52058336|1      |
|52046982|1      |
|52024601|1      |
|51094533|1      |
|51091186|1      |
|51062739|1      |
|51025426|1      |
|51025329|1      |
|51024250|1      |
|51021412|1      |
|50032151|1      |
+--------+-------+
only showing top 20 rows
Tabela 'dw.bridge_esc_etp' truncada antes da nova carga.
Tabela 'dw.bridge_esc_etp' carregada com 248,109 registros.
Tabela 'dw.bridge_esc_etp' carregada com 248,109 registros.


In [28]:
df_silver.unpersist()
spark.stop()
print("Sessão Spark finalizada.")

Sessão Spark finalizada.
